In [1]:
import os

In [2]:
%pwd

'c:\\projects\\SellWise\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\projects\\SellWise'

In [5]:
from dataclasses import dataclass
from pathlib import Path
from typing import List

@dataclass(frozen=True)
class EnsembleConfig:
    submission_dir: Path
    sample_submission_path: Path
    submission_files: List[str]
    final_submission_filename: str
    eval_row_offset: int

In [6]:
from SellWise.constants import *
from SellWise.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
        schema_filepath=SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([(self.config.artifacts_root)])

    def get_ensemble_config(self) -> EnsembleConfig:
        pred_config = self.config.predictor
        ingestion_config = self.config.data_ingestion

        create_directories([pred_config.submission_dir])

        # List of the 6 individual model prediction submission files
        submission_files = [
            "submission_recursive_store.csv",
            "submission_recursive_store_cat.csv",
            "submission_recursive_store_dept.csv",
            "submission_nonrecursive_store.csv",
            "submission_nonrecursive_store_cat.csv",
            "submission_nonrecursive_store_dept.csv"
        ]

        return EnsembleConfig(
            submission_dir=Path(pred_config.submission_dir),
            sample_submission_path=Path(ingestion_config.sample_submission_path),
            submission_files=submission_files,
            final_submission_filename="submission_final.csv",
            eval_row_offset=30490  # Evaluation row index split from original notebook
        )

In [8]:
import gc
import time
import os
import pandas as pd
import numpy as np
import warnings
from SellWise import logger

warnings.filterwarnings('ignore')

In [9]:
class EnsemblePredictor:
    """
    Combines the 6 individual model submission files (3 recursive + 3 non-recursive)
    by averaging their predictions across evaluation IDs, matching the original 
    3-1 Final Ensemble logic.
    """
    def __init__(self, config: EnsembleConfig):
        self.config = config

    def run(self):
        logger.info("Starting Ensemble Pipeline...")
        
        # Load sample submission and isolate evaluation IDs (rows 30490+)
        sample_sub = pd.read_csv(self.config.sample_submission_path)
        ids = pd.DataFrame({'id': sample_sub.iloc[self.config.eval_row_offset:]['id']})
        logger.info(f"Loaded evaluation target IDs: {len(ids)} rows")

        sub_dfs = []
        for file_name in self.config.submission_files:
            file_path = self.config.submission_dir / file_name
            
            # Strict file existence check
            if not file_path.exists():
                raise FileNotFoundError(f"Submission file not found at: {file_path}")

            logger.info(f"Loading submission: {file_path.name}")
            sub = pd.read_csv(file_path)
            
            # Merge with evaluation IDs on 'id' left join and set index
            sub_merged = ids.merge(sub, on='id', how='left').set_index('id')
            sub_dfs.append(sub_merged)

        # Calculate arithmetic mean across all 6 model predictions
        logger.info("Averaging 6 submission dataframes...")
        final_sub = sum(sub_dfs) / len(sub_dfs)

        # Export final submission CSV
        output_path = self.config.submission_dir / self.config.final_submission_filename
        final_sub.to_csv(output_path)
        logger.info(f"✅ Final ensemble successfully saved to: {output_path}")

        return final_sub

In [10]:
try:
    config = ConfigurationManager()
    ensemble_config = config.get_ensemble_config()
    predictor = EnsemblePredictor(config=ensemble_config)
    final_submission = predictor.run()
except Exception as e:
    raise e

[2026-09-24 22:22:25,912: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-09-24 22:22:25,936: INFO: common: yaml file: params.yaml loaded successfully]
[2026-09-24 22:22:25,946: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-09-24 22:22:25,948: INFO: common: created directory at: artifacts]
[2026-09-24 22:22:25,949: INFO: common: created directory at: artifacts/model_evaluation/submissions]
[2026-09-24 22:22:25,950: INFO: 3526482868: Starting Ensemble Pipeline...]
[2026-09-24 22:22:26,070: INFO: 3526482868: Loaded evaluation target IDs: 30490 rows]
[2026-09-24 22:22:26,071: INFO: 3526482868: Loading submission: submission_recursive_store.csv]
[2026-09-24 22:22:26,346: INFO: 3526482868: Loading submission: submission_recursive_store_cat.csv]
[2026-09-24 22:22:26,638: INFO: 3526482868: Loading submission: submission_recursive_store_dept.csv]
[2026-09-24 22:22:26,918: INFO: 3526482868: Loading submission: submission_nonrecursive_store.csv]
[2026-09-